# 01 – Audit data (CP1, FR-01)

Angka dihitung dari **gambar lokal**, lalu dibandingkan dengan Tabel 3.2 proposal (150 jinak / 60 ganas pasien; 7.657 / 3.487 citra). Selisih dilaporkan apa adanya. Dataset hanya dibaca.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.


## 1. Jalankan audit (dilewati bila `audit.csv` sudah ada)

In [ ]:
rc = run_module('src.audit', ['--config', CFG], skip_if_exists=P(cfg, 'cp1', 'audit.csv'))
print('kode keluar:', rc)

## 2. Perbandingan dengan Tabel 3.2 dan repo Kiani

In [ ]:
s = json.loads(P(cfg, 'cp1', 'audit_summary.json').read_text())
display(pd.DataFrame(s['perbandingan_referensi']))
print('ID pasien bertabrakan antar kelas:', s['id_bertabrakan_antar_kelas'])
print('Pasien label tidak konsisten:', s['pasien_label_tidak_konsisten'])
print('Pasien di >1 folder histopatologi:', s['pasien_di_lebih_dari_satu_folder_histo'])

## 3. Silang-periksa dengan `Patients_List_Updated_Final.xlsx`

In [ ]:
for k, v in s['xlsx'].items():
    print(k, ':', (v if not isinstance(v, list) or len(v) <= 15 else v[:15] + ['...']))

## 4. Resolusi dan syarat S7 (dihitung ulang dari gambar lokal)

In [ ]:
display(pd.DataFrame(s['s7']['per_resolusi']))
print('mayoritas:', s['s7'].get('mayoritas'))
print('minor gabungan:', s['s7'].get('minor_gabungan'))
print('pasien dengan >1 resolusi:', s['s7']['pasien_multi_resolusi'])
print('S7 dapat dijalankan:', s['s7'].get('s7_dapat_dijalankan'))

## 5. Pemeriksaan checkpoint CP1 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp1
report = finalize_cp1(cfg)
assert report['checkpoint'] == 'CP1'